# LeetCode #62 — Unique Paths: nhập môn Quy hoạch động trên lưới

**Mức độ:** Medium · **Chủ đề:** Math, Dynamic Programming, Combinatorics

Skeleton trên LeetCode:

```python
class Solution:
    def uniquePaths(self, m: int, n: int) -> int:
```

Mục tiêu của notebook là đi qua **cả con đường** từ đệ quy tới Quy hoạch động (DP):

1. **Đệ quy thuần** — đúng nhưng chậm theo hàm mũ. Vì sao?
2. **Đệ quy + ghi nhớ (memoization)** — "top-down DP".
3. **Bảng DP (tabulation)** — "bottom-up DP".
4. **Tối ưu bộ nhớ** còn một hàng.
5. **Công thức tổ hợp** `C(m + n - 2, m - 1)`.

# 1. Bài toán là gì?

Một robot đứng ở góc **trên trái** của lưới `m × n`. Mỗi bước nó chỉ được đi **sang phải** hoặc **xuống dưới**. Có bao nhiêu đường đi khác nhau tới góc **dưới phải**?

**Constraints (theo đề):**

- `1 <= m, n <= 100`
- Đáp án không vượt quá `2 * 10^9`.

---

## Ví dụ

```text
m = 3, n = 7  ->  28
m = 3, n = 2  ->  3      (Phải-Xuống-Xuống, Xuống-Phải-Xuống, Xuống-Xuống-Phải)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quy hoạch động (Dynamic Programming — DP)
2. DP hai chiều (bảng `dp[i][j]`)
3. Hoán vị, tổ hợp, tập con và thứ tự từ điển

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [1]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [2]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.2 DP hai chiều (bảng `dp[i][j]`)

### Khi nào cần bảng hai chiều?

Khi trạng thái cần **hai** thông số, thường gặp ở:

- **Lưới**: `dp[i][j]` = đáp án cho ô `(i, j)` (số đường đi, tổng nhỏ nhất…).
- **Hai chuỗi**: `dp[i][j]` = đáp án cho **tiền tố** `s[:i]` và `t[:j]` (edit distance, LCS, xen kẽ…).
- **Đoạn con**: `dp[i][j]` = đáp án cho đoạn `s[i..j]` (palindrome…).

### Ví dụ 1: số đường đi trên lưới

Robot ở góc trên-trái, mỗi bước đi **phải** hoặc **xuống**. Có bao nhiêu đường tới ô `(i, j)`? Bước cuối đến từ ô **trên** hoặc ô **trái**:

```text
dp[i][j] = dp[i-1][j] + dp[i][j-1]
hàng 0 và cột 0 đều = 1 (chỉ một cách: đi thẳng)
```

In [3]:
def so_duong(m, n):
    dp = [[0] * n for _ in range(m)]      # KHÔNG dùng [[0] * n] * m
    for i in range(m):
        for j in range(n):
            if i == 0 or j == 0:
                dp[i][j] = 1
            else:
                dp[i][j] = dp[i - 1][j] + dp[i][j - 1]
    return dp

for row in so_duong(3, 5):
    print(" ".join(f"{v:>3}" for v in row))

  1   1   1   1   1
  1   2   3   4   5
  1   3   6  10  15


Thứ tự điền: theo hàng từ trên xuống, trong hàng từ trái sang → khi tính `dp[i][j]`, ô trên và ô trái **đã có**.

**Tiết kiệm bộ nhớ:** hàng `i` chỉ cần hàng `i−1` → giữ một mảng một chiều `row`, cập nhật `row[j] += row[j-1]` (giá trị cũ của `row[j]` chính là ô trên).

In [4]:
def so_duong_mot_hang(m, n):
    row = [1] * n
    for _ in range(1, m):
        for j in range(1, n):
            row[j] += row[j - 1]          # row[j] (cũ) = trên, row[j-1] (mới) = trái
    return row[-1]
print("so_duong_mot_hang(3, 7) =", so_duong_mot_hang(3, 7), "| so_duong(3, 7) =", so_duong(3, 7)[-1][-1])

so_duong_mot_hang(3, 7) = 28 | so_duong(3, 7) = 28


### Ví dụ 2: hai chuỗi — dãy con chung dài nhất (LCS)

`dp[i][j]` = độ dài dãy con chung dài nhất của `s[:i]` và `t[:j]`. Xét ký tự **cuối** của hai tiền tố:

- `s[i-1] == t[j-1]` → dùng nó: `dp[i][j] = dp[i-1][j-1] + 1`.
- khác → bỏ ký tự cuối của một trong hai: `dp[i][j] = max(dp[i-1][j], dp[i][j-1])`.

Hàng 0 và cột 0 ứng với **chuỗi rỗng** → bằng 0. Đó là lý do bảng có kích thước `(n+1) × (m+1)`.

In [5]:
def lcs(s, t):
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp

s_demo, t_demo = "abcde", "ace"
bang = lcs(s_demo, t_demo)
print("      ''  " + "   ".join(t_demo))
for i, row in enumerate(bang):
    print(f"{('' if i == 0 else s_demo[i - 1])!r:>4} " + " ".join(f"{v:>3}" for v in row))
print("LCS =", bang[-1][-1])

      ''  a   c   e
  ''   0   0   0   0
 'a'   0   1   1   1
 'b'   0   1   1   1
 'c'   0   1   2   2
 'd'   0   1   2   2
 'e'   0   1   2   3
LCS = 3


### Ví dụ 3: khoảng cách sửa (edit distance) — số phép chèn/xóa/thay ít nhất

`dp[i][j]` = số phép sửa ít nhất biến `s[:i]` thành `t[:j]`.

- Ký tự cuối bằng nhau → `dp[i-1][j-1]`.
- Khác → `1 + min(thay: dp[i-1][j-1], xóa s[i-1]: dp[i-1][j], chèn t[j-1]: dp[i][j-1])`.
- Cơ sở: `dp[i][0] = i` (xóa hết), `dp[0][j] = j` (chèn hết).

In [6]:
def khoang_cach_sua(s, t):
    n, m = len(s), len(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i - 1][j], dp[i][j - 1])
    return dp[n][m]

for a_, b_ in [("horse", "ros"), ("kitten", "sitting"), ("", "abc")]:
    print(f"{a_!r} → {b_!r}: {khoang_cach_sua(a_, b_)}")

'horse' → 'ros': 3
'kitten' → 'sitting': 3
'' → 'abc': 3


### Lỗi hay gặp

- `[[0] * n] * m` tạo `m` hàng là **cùng một** list → sửa một ô thì cả cột đổi.
- Lệch chỉ số: bảng `(n+1)×(m+1)` nhưng ký tự là `s[i-1]`, `t[j-1]`.
- Quên hàng/cột cơ sở (chuỗi rỗng, biên lưới).
- Khi tối ưu một hàng: ghi đè giá trị cần dùng lại (ví dụ `dp[i-1][j-1]`) — phải lưu tạm trước.

In [7]:
sai = [[0] * 3] * 2
sai[0][0] = 9
dung = [[0] * 3 for _ in range(2)]
dung[0][0] = 9
print("[[0]*3]*2 sau khi gán ô (0,0):", sai, "| tạo bằng vòng for:", dung)

[[0]*3]*2 sau khi gán ô (0,0): [[9, 0, 0], [9, 0, 0]] | tạo bằng vòng for: [[9, 0, 0], [0, 0, 0]]


## 2.3 Hoán vị, tổ hợp, tập con và thứ tự từ điển

### Đếm

| Đối tượng | Ý nghĩa | Số lượng |
|---|---|---|
| Hoán vị của `n` phần tử | sắp **thứ tự** tất cả | `n!` |
| Chỉnh hợp chập `k` | chọn `k` phần tử **có thứ tự** | `n! / (n−k)!` |
| Tổ hợp chập `k` | chọn `k` phần tử **không** thứ tự | `C(n, k) = n! / (k! (n−k)!)` |
| Tập con | mỗi phần tử lấy/không | `2^n` |
| Hoán vị có lặp | `n` phần tử, nhóm trùng cỡ `c1, c2, …` | `n! / (c1! c2! …)` |

In [8]:
from math import factorial, comb, perm
from itertools import permutations, combinations, product

print("4! =", factorial(4), "| P(4,2) =", perm(4, 2), "| C(4,2) =", comb(4, 2), "| 2^4 =", 2 ** 4)
print("hoán vị [1,2,3]:", list(permutations([1, 2, 3])))
print("tổ hợp chập 2 của [1,2,3,4]:", list(combinations([1, 2, 3, 4], 2)))
print("hoán vị khác nhau của 'aab':", sorted(set(permutations("aab"))), "=", factorial(3) // factorial(2))
print("tích Descartes {0,1}^2:", list(product([0, 1], repeat=2)))

4! = 24 | P(4,2) = 12 | C(4,2) = 6 | 2^4 = 16
hoán vị [1,2,3]: [(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
tổ hợp chập 2 của [1,2,3,4]: [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
hoán vị khác nhau của 'aab': [('a', 'a', 'b'), ('a', 'b', 'a'), ('b', 'a', 'a')] = 3
tích Descartes {0,1}^2: [(0, 0), (0, 1), (1, 0), (1, 1)]


### Thứ tự từ điển (lexicographic order)

So hai dãy như so từ trong từ điển: so phần tử **đầu tiên khác nhau**; nếu một dãy là tiền tố của dãy kia, dãy ngắn hơn đứng trước. Python so sánh list/tuple/str đúng theo kiểu này.

Liệt kê hoán vị của `[1, 2, 3]` theo thứ tự từ điển: `123, 132, 213, 231, 312, 321`.

**Quan sát về hoán vị kế tiếp:** đi từ phải sang, phần **hậu tố giảm dần** dài nhất là hậu tố "đã lớn nhất có thể". Muốn hoán vị lớn hơn kế tiếp, phải tăng phần tử **ngay trước** hậu tố đó lên một chút.

In [9]:
print([1, 2, 3] < [1, 3], "|", "abc" < "abd", "|", [1, 2] < [1, 2, 0], "|", "10" < "9")
for p in permutations([1, 2, 3]):
    i = len(p) - 1
    while i > 0 and p[i - 1] >= p[i]:
        i -= 1
    print(p, "| hậu tố giảm dần dài nhất:", p[i:])

True | True | True | True
(1, 2, 3) | hậu tố giảm dần dài nhất: (3,)
(1, 3, 2) | hậu tố giảm dần dài nhất: (3, 2)
(2, 1, 3) | hậu tố giảm dần dài nhất: (3,)
(2, 3, 1) | hậu tố giảm dần dài nhất: (3, 1)
(3, 1, 2) | hậu tố giảm dần dài nhất: (2,)
(3, 2, 1) | hậu tố giảm dần dài nhất: (3, 2, 1)


### Liên hệ với quay lui

`itertools` sinh sẵn hoán vị/tổ hợp, nhưng phỏng vấn thường yêu cầu **tự sinh** bằng quay lui (để kèm cắt tỉa, bỏ trùng, điều kiện riêng). Kích thước kết quả (`n!`, `C(n,k)`, `2^n`) cho biết độ phức tạp tối thiểu — không thuật toán nào liệt kê nhanh hơn số lời giải.

## 2.4 Áp dụng vào bài này

Đây chính là ví dụ **DP trên lưới** ở trên: `dp[i][j] = dp[i−1][j] + dp[i][j−1]`, có thể rút về **một hàng**. Cách toán học: mọi đường đi gồm `m−1` bước xuống và `n−1` bước sang phải theo thứ tự bất kỳ → **tổ hợp** `C(m+n−2, m−1)`.

# 3. Ý tưởng then chốt: tách theo bước cuối

Muốn tới ô `(r, c)`, bước **cuối cùng** chỉ có thể đến từ:

- ô **phía trên** `(r - 1, c)` (đi xuống), hoặc
- ô **bên trái** `(r, c - 1)` (đi sang phải).

Hai nhóm đường này **không trùng nhau** (khác bước cuối). Vậy:

```text
paths(r, c) = paths(r - 1, c) + paths(r, c - 1)
```

**Trường hợp cơ sở:** mọi ô ở **hàng 0** hoặc **cột 0** chỉ có **1** đường (đi thẳng).

# 4. Cách 1: đệ quy thuần

In [10]:
def paths_recursive(r: int, c: int) -> int:
    if r == 0 or c == 0:
        return 1
    return paths_recursive(r - 1, c) + paths_recursive(r, c - 1)

def unique_paths_recursive(m: int, n: int) -> int:
    return paths_recursive(m - 1, n - 1)

print(unique_paths_recursive(3, 7), unique_paths_recursive(3, 2))

28 3


## Vì sao chậm?

Cùng một ô được tính **rất nhiều lần**. Đếm số lần gọi:

In [11]:
calls = 0
def paths_count_calls(r, c):
    global calls
    calls += 1
    if r == 0 or c == 0:
        return 1
    return paths_count_calls(r - 1, c) + paths_count_calls(r, c - 1)

for m, n in [(3, 7), (8, 8), (12, 12)]:
    calls = 0
    result = paths_count_calls(m - 1, n - 1)
    print(f"m={m:>2}, n={n:>2}: đáp án {result:>7}, số lần gọi hàm = {calls:,}")

m= 3, n= 7: đáp án      28, số lần gọi hàm = 55
m= 8, n= 8: đáp án    3432, số lần gọi hàm = 6,863


m=12, n=12: đáp án  705432, số lần gọi hàm = 1,410,863

Số lần gọi tăng **theo hàm mũ**. Với `100 × 100` là bất khả thi.

Nhưng chỉ có `m × n` ô **khác nhau**! Nếu mỗi ô chỉ tính **một lần** thì rất nhanh.

# 5. Cách 2: đệ quy + ghi nhớ (memoization)

Lưu kết quả đã tính vào `dict` (hoặc dùng `functools.lru_cache`). Lần sau gặp lại thì trả luôn.

In [12]:
from functools import lru_cache

def unique_paths_memo(m: int, n: int) -> int:
    memo = {}

    def paths(r: int, c: int) -> int:
        if r == 0 or c == 0:
            return 1
        if (r, c) in memo:
            return memo[(r, c)]
        memo[(r, c)] = paths(r - 1, c) + paths(r, c - 1)
        return memo[(r, c)]

    return paths(m - 1, n - 1)


def unique_paths_cache(m: int, n: int) -> int:
    @lru_cache(maxsize=None)
    def paths(r, c):
        if r == 0 or c == 0:
            return 1
        return paths(r - 1, c) + paths(r, c - 1)
    return paths(m - 1, n - 1)

print(unique_paths_memo(3, 7), unique_paths_memo(50, 50))
print(unique_paths_cache(3, 7))

28 25477612258980856902730428600
28


`@lru_cache` là một **decorator**: nó "bọc" hàm để tự động ghi nhớ kết quả theo tham số. Gọn, nhưng nên hiểu bản `memo` tự viết trước.

**Lưu ý:** đệ quy sâu `m + n` tầng. Với `100 × 100` là ~200 tầng — trong giới hạn mặc định của Python (~1000), nhưng nếu lưới lớn hơn nhiều thì sẽ `RecursionError`. Đó là một lý do dùng bảng DP.

# 6. Cách 3: bảng DP (bottom-up)

Tạo bảng `dp[r][c]` = số đường tới ô `(r, c)`. Điền theo thứ tự **từ trên xuống, trái sang phải**, để khi tính `dp[r][c]` thì `dp[r-1][c]` và `dp[r][c-1]` đã có.

```text
m = 3, n = 7:
1  1  1  1  1  1  1
1  2  3  4  5  6  7
1  3  6 10 15 21 28
```

Mỗi ô = ô trên + ô trái.

In [13]:
from typing import List

def unique_paths_table(m: int, n: int) -> int:
    dp = [[1] * n for _ in range(m)]          # hàng 0 và cột 0 đều là 1
    for r in range(1, m):
        for c in range(1, n):
            dp[r][c] = dp[r - 1][c] + dp[r][c - 1]
    return dp[m - 1][n - 1]

def show_table(m, n):
    dp = [[1] * n for _ in range(m)]
    for r in range(1, m):
        for c in range(1, n):
            dp[r][c] = dp[r - 1][c] + dp[r][c - 1]
    w = len(str(dp[-1][-1]))
    for row in dp:
        print(" ".join(str(x).rjust(w) for x in row))

show_table(3, 7)
print(unique_paths_table(3, 7), unique_paths_table(3, 2))

 1  1  1  1  1  1  1
 1  2  3  4  5  6  7
 1  3  6 10 15 21 28
28 3


Nhìn bảng có quen không? Đó là **tam giác Pascal** đặt nghiêng!

# 7. Cách 4: chỉ giữ một hàng

`dp[r][c]` chỉ cần **hàng trên** (`dp[r-1][c]`) và **ô vừa tính bên trái** (`dp[r][c-1]`). Dùng một list `row` dài `n`:

- Trước khi cập nhật, `row[c]` đang là giá trị của **hàng trên**.
- `row[c - 1]` đã là giá trị **hàng hiện tại** (vừa cập nhật).

Vậy `row[c] += row[c - 1]` là đủ!

In [14]:
def unique_paths_1d(m: int, n: int) -> int:
    row = [1] * n
    for _ in range(1, m):
        for c in range(1, n):
            row[c] += row[c - 1]
    return row[-1]

print(unique_paths_1d(3, 7), unique_paths_1d(3, 2))

28 3


# 8. Cách 5: công thức tổ hợp

Mọi đường đi đều gồm đúng `m - 1` bước **Xuống** và `n - 1` bước **Phải**, tổng `m + n - 2` bước. Một đường đi = chọn **vị trí** của `m - 1` bước Xuống trong `m + n - 2` bước:

```text
đáp án = C(m + n - 2, m - 1)
```

Ví dụ `m = 3, n = 7`: `C(8, 2) = 28`. ✓

In [15]:
from math import comb

def unique_paths_comb(m: int, n: int) -> int:
    return comb(m + n - 2, m - 1)

print(unique_paths_comb(3, 7), unique_paths_comb(3, 2))

28 3


Nếu không dùng `math.comb`, có thể tính `C(N, k)` tăng dần mà vẫn luôn là số nguyên:

```python
result = 1
for i in range(1, k + 1):
    result = result * (N - k + i) // i
```

Sau bước `i`, `result = C(N - k + i, i)` — luôn nguyên, nên phép `//` chính xác.

# 9. Phiên bản đúng format LeetCode

In [16]:
class Solution:
    def uniquePaths(self, m: int, n: int) -> int:
        row = [1] * n
        for _ in range(1, m):
            for c in range(1, n):
                row[c] += row[c - 1]
        return row[-1]

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS  {actual}")


sol = Solution()
check("Ví dụ 1: 3 x 7", sol.uniquePaths(3, 7), 28)
check("Ví dụ 2: 3 x 2", sol.uniquePaths(3, 2), 3)
check("1 x 1", sol.uniquePaths(1, 1), 1)
check("1 x 100 (một hàng)", sol.uniquePaths(1, 100), 1)
check("7 x 3 (đối xứng)", sol.uniquePaths(7, 3), 28)
check("Lớn nhất theo đề", sol.uniquePaths(23, 12), comb(33, 11))

for m in range(1, 16):
    for n in range(1, 16):
        expected = unique_paths_comb(m, n)
        assert sol.uniquePaths(m, n) == expected
        assert unique_paths_table(m, n) == expected
        assert unique_paths_memo(m, n) == expected
        if m + n <= 16:
            assert unique_paths_recursive(m, n) == expected
print("Mọi m, n trong 1..15: các cách đều khớp công thức tổ hợp")

Ví dụ 1: 3 x 7         PASS  28
Ví dụ 2: 3 x 2         PASS  3
1 x 1                  PASS  1
1 x 100 (một hàng)     PASS  1
7 x 3 (đối xứng)       PASS  28
Lớn nhất theo đề       PASS  193536720
Mọi m, n trong 1..15: các cách đều khớp công thức tổ hợp


# 10. Vì sao đúng? Độ phức tạp?

**Đúng:** mỗi đường tới `(r, c)` có bước cuối hoặc từ trên, hoặc từ trái — hai tập rời nhau và phủ hết → công thức cộng đúng. Ô ở hàng 0 / cột 0 chỉ có một đường. Quy nạp theo `r + c` cho ra mọi ô.

| Cách | Thời gian | Bộ nhớ |
|---|---|---|
| Đệ quy thuần | ~O(2^(m+n)) | O(m + n) |
| Memoization | O(m · n) | O(m · n) |
| Bảng DP | O(m · n) | O(m · n) |
| Một hàng | O(m · n) | O(n) |
| Tổ hợp | O(min(m, n)) | O(1) |

# 11. Những lỗi dễ mắc

## Lỗi 1: Khởi tạo hàng/cột đầu bằng 0

Toàn bảng thành 0 (không có đường nào "bắt đầu").

In [18]:
def wrong_init_zero(m: int, n: int) -> int:
    dp = [[0] * n for _ in range(m)]
    dp[0][0] = 1
    for r in range(1, m):                      # SAI: bỏ qua hàng 0 và cột 0
        for c in range(1, n):
            dp[r][c] = dp[r - 1][c] + dp[r][c - 1]
    return dp[m - 1][n - 1]

print("Sai :", wrong_init_zero(3, 7), "| Đúng:", sol.uniquePaths(3, 7))

Sai : 0 | Đúng: 28


Có thể khởi tạo bằng 0 **nếu** vòng lặp duyệt cả hàng 0 và cột 0, chỉ cộng các ô hợp lệ. Cách đơn giản hơn: điền 1 cho hàng 0 và cột 0.

## Lỗi 2: Nhầm `C(m + n, m)` với `C(m + n - 2, m - 1)`

Số **bước** là `(m - 1) + (n - 1)`, không phải `m + n`.

## Lỗi 3: Duyệt bản 1D theo chiều ngược

`for c in range(n - 1, 0, -1): row[c] += row[c - 1]` dùng `row[c - 1]` **của hàng trên** (chưa cập nhật) → sai. Bản 1D phải duyệt **trái sang phải**.

In [19]:
def wrong_1d_reverse(m: int, n: int) -> int:
    row = [1] * n
    for _ in range(1, m):
        for c in range(n - 1, 0, -1):          # SAI chiều
            row[c] += row[c - 1]
    return row[-1]

print("Sai :", wrong_1d_reverse(3, 7), "| Đúng:", sol.uniquePaths(3, 7))

Sai : 4 | Đúng: 28


## Lỗi 4: Đệ quy không ghi nhớ

Chạy được với lưới nhỏ, nhưng `TLE` với `m, n` cỡ 20 trở lên.

# 12. Mẹo nhận diện pattern

**DP trên lưới:** "đếm số đường / tìm đường tốt nhất, chỉ đi phải hoặc xuống".

1. Định nghĩa `dp[r][c]` = đáp án cho bài toán con kết thúc tại `(r, c)`.
2. Viết công thức từ các ô **có thể đi tới** `(r, c)`.
3. Xác định trường hợp cơ sở (hàng/cột đầu).
4. Chọn thứ tự điền sao cho phụ thuộc đã tính xong.
5. Tối ưu bộ nhớ nếu chỉ phụ thuộc hàng trước.

Bài cùng họ: **#63 Unique Paths II** (có vật cản), **#64 Minimum Path Sum**, **#120 Triangle**, **#931 Minimum Falling Path Sum**, **#174 Dungeon Game** (Hard).

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Lập bảng DP cho `m = 4`, `n = 4`. Kiểm tra bằng `C(6, 3)`.

## Bài 2 — Điền chỗ trống (bảng DP)

In [20]:
def exercise_2_unique_paths(m: int, n: int) -> int:
    dp = [[1] * n for _ in range(m)]
    # TODO: điền dp[r][c] = dp[r-1][c] + dp[r][c-1] cho r >= 1, c >= 1
    return dp[m - 1][n - 1]


# assert exercise_2_unique_paths(3, 7) == 28
# assert exercise_2_unique_paths(3, 2) == 3
# assert exercise_2_unique_paths(1, 1) == 1

## Bài 3 — Tính tổ hợp không dùng `math.comb`

Viết `n_choose_k(N, k)` chỉ dùng số nguyên (công thức tăng dần ở mục 8).

```python
n_choose_k(8, 2)   -> 28
n_choose_k(10, 0)  -> 1
n_choose_k(198, 99) == math.comb(198, 99)
```

In [21]:
def exercise_3_n_choose_k(N: int, k: int) -> int:
    # TODO
    pass


# assert exercise_3_n_choose_k(8, 2) == 28
# assert exercise_3_n_choose_k(10, 0) == 1
# assert exercise_3_n_choose_k(198, 99) == comb(198, 99)

## Bài 4 — Biến thể: được đi thêm chéo

Robot được đi **phải**, **xuống**, hoặc **chéo xuống-phải**. Đếm số đường (số Delannoy).

```python
paths_with_diagonal(2, 2) -> 3       # P-X, X-P, Chéo
paths_with_diagonal(3, 3) -> 13
```

In [22]:
def exercise_4_paths_with_diagonal(m: int, n: int) -> int:
    # TODO
    pass


# assert exercise_4_paths_with_diagonal(2, 2) == 3
# assert exercise_4_paths_with_diagonal(3, 3) == 13
# assert exercise_4_paths_with_diagonal(1, 5) == 1

## Bài 5 — Mini challenge: liệt kê đường đi

Liệt kê **mọi** đường đi dưới dạng chuỗi gồm `'R'` (phải) và `'D'` (xuống), theo thứ tự từ điển. Kiểm tra số lượng khớp `uniquePaths`.

```python
list_paths(2, 3) -> ["DRR", "RDR", "RRD"]
```

In [23]:
def exercise_5_list_paths(m: int, n: int) -> List[str]:
    # TODO: backtracking với số bước D và R còn lại
    pass


# assert exercise_5_list_paths(2, 3) == ["DRR", "RDR", "RRD"]
# assert len(exercise_5_list_paths(4, 4)) == 20

# 14. Gợi ý

**Bài 1**
- Mức 1: hàng 1 là `1 2 3 4`, hàng 2 là `1 3 6 10`.

**Bài 3**
- Mức 1: `k = min(k, N - k)` để vòng lặp ngắn hơn.
- Mức 2: `result = result * (N - k + i) // i` với `i = 1..k`.

**Bài 4**
- Mức 1: `dp[r][c] = dp[r-1][c] + dp[r][c-1] + dp[r-1][c-1]`.

**Bài 5**
- Mức 1: `backtrack(d_left, r_left)`; thử `'D'` trước `'R'` để có thứ tự từ điển.

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [24]:
show_table(4, 4)
print("C(6, 3) =", comb(6, 3))

 1  1  1  1
 1  2  3  4
 1  3  6 10
 1  4 10 20
C(6, 3) = 20


## Đáp án Bài 2

In [25]:
def answer_2_unique_paths(m: int, n: int) -> int:
    dp = [[1] * n for _ in range(m)]
    for r in range(1, m):
        for c in range(1, n):
            dp[r][c] = dp[r - 1][c] + dp[r][c - 1]
    return dp[m - 1][n - 1]

assert answer_2_unique_paths(3, 7) == 28
assert answer_2_unique_paths(3, 2) == 3
assert answer_2_unique_paths(1, 1) == 1
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [26]:
def answer_3_n_choose_k(N: int, k: int) -> int:
    k = min(k, N - k)
    result = 1
    for i in range(1, k + 1):
        result = result * (N - k + i) // i
    return result

assert answer_3_n_choose_k(8, 2) == 28
assert answer_3_n_choose_k(10, 0) == 1
assert answer_3_n_choose_k(198, 99) == comb(198, 99)
assert all(answer_3_n_choose_k(N, k) == comb(N, k) for N in range(40) for k in range(N + 1))
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [27]:
def answer_4_paths_with_diagonal(m: int, n: int) -> int:
    dp = [[1] * n for _ in range(m)]
    for r in range(1, m):
        for c in range(1, n):
            dp[r][c] = dp[r - 1][c] + dp[r][c - 1] + dp[r - 1][c - 1]
    return dp[m - 1][n - 1]

assert answer_4_paths_with_diagonal(2, 2) == 3
assert answer_4_paths_with_diagonal(3, 3) == 13
assert answer_4_paths_with_diagonal(1, 5) == 1
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [28]:
def answer_5_list_paths(m: int, n: int) -> List[str]:
    result, path = [], []

    def backtrack(d_left: int, r_left: int):
        if d_left == 0 and r_left == 0:
            result.append("".join(path))
            return
        if d_left:
            path.append("D"); backtrack(d_left - 1, r_left); path.pop()
        if r_left:
            path.append("R"); backtrack(d_left, r_left - 1); path.pop()

    backtrack(m - 1, n - 1)
    return result

assert answer_5_list_paths(2, 3) == ["DRR", "RDR", "RRD"]
assert len(answer_5_list_paths(4, 4)) == 20
assert all(len(answer_5_list_paths(m, n)) == sol.uniquePaths(m, n) for m in range(1, 6) for n in range(1, 6))
print("Bài 5 PASS")

Bài 5 PASS


So sánh: **liệt kê** tốn thời gian tỉ lệ với số kết quả (có thể khổng lồ), còn **đếm** bằng DP chỉ `O(m · n)`. Khi đề chỉ hỏi "bao nhiêu", hãy nghĩ tới DP.

# 16. Tổng kết

> Số đường tới một ô = số đường tới ô **trên** + số đường tới ô **trái**. Điền bảng từ trên xuống, trái sang phải (hoặc chỉ giữ một hàng). Đáp án cũng bằng `C(m + n - 2, m - 1)`.

---

## Checklist trước khi nộp

- [ ] Hàng 0 và cột 0 khởi tạo bằng 1?
- [ ] Thứ tự điền: trên xuống, trái sang phải?
- [ ] Bản 1D duyệt `c` tăng dần?
- [ ] Đã thử `1 × 1`, một hàng, một cột?